In [5]:
import os
import shutil
import cv2
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from concurrent.futures import ThreadPoolExecutor, as_completed

# Prevenim blocarea thread-urilor OpenCV
cv2.setNumThreads(0)

# ==========================================
# 1. CONFIGURARE CAI 
# ==========================================
# Seteaza caile catre fisierele tale
CSV_PATH = Path(r"B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\originals\EyePACS\original_train_labels.csv") # Modifica cu CSV-ul tau combinat sau EyePACS
IMG_FOLDER = Path(r"B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\originals\EyePACS\Images")

OUTPUT_DIR = Path(r"B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\processed_by_me\eyepacs\eyepacs_cleaned")

CLASSES = ["0", "1", "2", "3", "4"]
WORKERS = max((os.cpu_count() or 2) - 1, 1)

RATIO_TRAIN, RATIO_VAL, RATIO_TEST = 0.70, 0.20, 0.10

# ==========================================
# 2. FUNCTIILE DE CURATARE (Din cerinta ta)
# ==========================================
THRESHOLDS = {
    "0": {
        "blur_min": 55.0,        
        "bright_min": 15.0,      
        "bright_max": 180.0,     
        "area_min": 0.25,        
        "area_max": 0.95,        
        "circularity_min": 0.80, 
        "glare_max_ratio": 0.005 
    },
    "minoritare": {
        "blur_min": 15.0,       
        "bright_min": 8.0,      
        "bright_max": 220.0,    
        "area_min": 0.15,       
        "area_max": 0.98,       
        "circularity_min": 0.60,  
        "glare_max_ratio": 0.05   
    }
}

In [6]:
def evaluate_image(image_path, cls):
    img = cv2.imread(str(image_path))
    if img is None: return False, "Eroare_Citire"

    rules = THRESHOLDS["0"] if str(cls) == "0" else THRESHOLDS["minoritare"]

    img_resized = cv2.resize(img, (512, 512))
    gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
    total_pixels = 512 * 512

    non_black_pixels = gray[gray > 10]
    if len(non_black_pixels) == 0: return False, "Imagine_Neagra"
        
    mean_brightness = np.mean(non_black_pixels)
    if mean_brightness < rules["bright_min"]: return False, "Prea_Intunecata"
    if mean_brightness > rules["bright_max"]: return False, "Supraexpusa_Total"

    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    if laplacian_var < rules["blur_min"]: return False, "Blurata"

    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    kernel_clean = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    thresh_clean = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_clean)
    contours, _ = cv2.findContours(thresh_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours: return False, "Fara_Contur"

    contur_ochi = max(contours, key=cv2.contourArea)
    ochi_area = cv2.contourArea(contur_ochi)
    
    area_ratio = ochi_area / total_pixels
    if area_ratio < rules["area_min"]: return False, "Zoom_Prea_Mic"
    if area_ratio > rules["area_max"]: return False, "Zoom_Exagerat_Taiat"

    perimeter = cv2.arcLength(contur_ochi, True)
    if perimeter == 0: return False, "Eroare_Geometrie"
        
    circularity = (4 * np.pi * ochi_area) / (perimeter * perimeter)
    if circularity < rules["circularity_min"]: return False, "Forma_Taiata_Neregulata"

    glare_mask = gray > 245
    glare_ratio = np.sum(glare_mask) / (ochi_area + 1e-6) 
    
    if glare_ratio > rules["glare_max_ratio"]: return False, "Reflexie_Blit_Lentila"

    return True, "OK"

def process_evaluation_task(task):
    img_path, label = task
    is_valid, reason = evaluate_image(img_path, label)
    return img_path, label, is_valid, reason

def copy_file(task):
    src, dst = task
    try:
        shutil.copy2(src, dst)
        return True
    except: return False

In [7]:
# ==========================================
# 3. EXECUTIA PRINCIPALA
# ==========================================
def main():
    if OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
    for split in ['train', 'val', 'test']:
        for cls in CLASSES:
            (OUTPUT_DIR / split / cls).mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(CSV_PATH)
    # Autodetect columns
    img_col, lbl_col = ('id_code', 'diagnosis') if 'id_code' in df.columns else ('image', 'level')

    print("\n--- 1. Cautarea imaginilor pe disc ---")
    fisiere_existente = {f.name: f for f in IMG_FOLDER.iterdir() if f.is_file()}
    
    tasks_eval = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Asociere fisier"):
        img_id, label = str(row[img_col]), str(row[lbl_col])
        for ext in ['.jpeg', '.jpg', '.png', '.tif']:
            nume = f"{img_id}{ext}"
            if nume in fisiere_existente:
                tasks_eval.append((fisiere_existente[nume], label))
                break

    print(f"\n--- 2. Curatarea imaginilor ({len(tasks_eval)} imagini gasite) ---")
    valid_data = []
    stats_respinse = {}
    
    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        for result in tqdm(executor.map(process_evaluation_task, tasks_eval), total=len(tasks_eval), desc="Evaluare Filtre"):
            img_path, label, is_valid, reason = result
            if is_valid:
                valid_data.append({'image_path': img_path, 'label': label})
            else:
                stats_respinse[reason] = stats_respinse.get(reason, 0) + 1

    df_valid = pd.DataFrame(valid_data)
    print(f"\n✅ Imagini ramase dupa curatare: {len(df_valid)}")
    print("Motive respingere:")
    for motiv, count in sorted(stats_respinse.items(), key=lambda x: x[1], reverse=True):
        print(f"  - {motiv}: {count} imagini")

    if len(df_valid) == 0: return

    print("\n--- 3. Impartirea Stratificata ---")
    df_train_val, df_test = train_test_split(df_valid, test_size=RATIO_TEST, stratify=df_valid['label'], random_state=42)
    val_fraction = RATIO_VAL / (RATIO_TRAIN + RATIO_VAL)
    df_train, df_val = train_test_split(df_train_val, test_size=val_fraction, stratify=df_train_val['label'], random_state=42)

    print(f"Distributie: TRAIN: {len(df_train)} | VAL: {len(df_val)} | TEST: {len(df_test)}")

    tasks_copy = []
    def add_copy_tasks(dataframe, split_name):
        for _, row in dataframe.iterrows():
            src = row['image_path']
            dst = OUTPUT_DIR / split_name / row['label'] / src.name
            tasks_copy.append((src, dst))

    add_copy_tasks(df_train, 'train')
    add_copy_tasks(df_val, 'val')
    add_copy_tasks(df_test, 'test')

    print("\n--- 4. Copierea in noua structura ---")
    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        for _ in tqdm(executor.map(copy_file, tasks_copy), total=len(tasks_copy), desc="Copiere"): pass

    print(f"\nPASUL 1 FINALIZAT! Setul curat se afla in: {OUTPUT_DIR}")

In [8]:
if __name__ == '__main__':
    main()


--- 1. Cautarea imaginilor pe disc ---


Asociere fisier:   0%|          | 0/35126 [00:00<?, ?it/s]


--- 2. Curatarea imaginilor (35126 imagini gasite) ---


Evaluare Filtre:   0%|          | 0/35126 [00:00<?, ?it/s]


✅ Imagini ramase dupa curatare: 27529
Motive respingere:
  - Blurata: 5714 imagini
  - Reflexie_Blit_Lentila: 840 imagini
  - Forma_Taiata_Neregulata: 687 imagini
  - Supraexpusa_Total: 334 imagini
  - Prea_Intunecata: 9 imagini
  - Zoom_Prea_Mic: 8 imagini
  - Imagine_Neagra: 2 imagini
  - Zoom_Exagerat_Taiat: 2 imagini
  - Eroare_Citire: 1 imagini

--- 3. Impartirea Stratificata ---
Distributie: TRAIN: 19270 | VAL: 5506 | TEST: 2753

--- 4. Copierea in noua structura ---


Copiere:   0%|          | 0/27529 [00:00<?, ?it/s]


PASUL 1 FINALIZAT! Setul curat se afla in: B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\processed_by_me\eyepacs\eyepacs_cleaned
